In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import statsmodels.api as sm

insurance_df = pd.read_csv("../Data/insurance.csv")

In [2]:
insurance_df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


### Data Splitting

In [ ]:
from sklearn.model_selection import train_test_split

insurance_df["smoker_flag"] = np.where(insurance_df["smoker"] == "yes", 1, 0)

features = ["age", "bmi", "children", "smoker_flag"]

X = sm.add_constant(insurance_df[features])
y = insurance_df["charges"]
#1 step
X, X_test, y, y_test = train_test_split(X, y, test_size=.2, random_state=2023)
#2 step
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=.25, random_state=2024)

In [18]:
y_valid

202     13012.20865
385      1261.85900
715     12146.97100
996      7418.52200
904     12644.58900
           ...     
315      9722.76950
1263     7337.74800
476     35147.52848
253      4260.74400
492      2196.47320
Name: charges, Length: 268, dtype: float64

Duomenų padalijimas į train ir test.

Kategorinis smoker perdirbamas kaip  dvejetainis kintamasis smoker_flag, kuris nurodo, ar asmuo rūko. Tada parenkami požymiai (age, bmi, children, smoker_flag) ir pridedama konstanta regresijos modeliui.

1 step. Naudojant train_test_split funkciją, duomenys padalijami taip, kad 80 % būtų naudojami mokymui, o 20 % – testavimui. random_state užtikrina, kad padalijimas būtų pakartojamas.

2 step. Duomenys papildomai padalijami į training ir validation dalis. Tai leidžia modelį treniruoti su viena duomenų dalimi ir tikrinti jo veikimą su atskiru validacijos rinkiniu prieš galutinį testavimą.

### Validation Scoring

In [19]:
model = sm.OLS(y_train, X_train).fit()

model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                charges   R-squared:                       0.756
Model:                            OLS   Adj. R-squared:                  0.755
Method:                 Least Squares   F-statistic:                     659.1
Date:                Wed, 25 Feb 2026   Prob (F-statistic):          7.16e-259
Time:                        15:04:08   Log-Likelihood:                -8661.8
No. Observations:                 856   AIC:                         1.733e+04
Df Residuals:                     851   BIC:                         1.736e+04
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
===============================================================================
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const       -1.294e+04   1174.920    -11.017      0.000   -1.52e+04   -1.06e+04
age           271.7960     14.851     18.301      0.000     242.647     300.945
bmi           328.6062     33.523      9.802      0.000     262.809     394.403
children      337.2685    172.429      1.956      0.051      -1.167     675.704
smoker_flag  2.386e+04    507.386     47.016      0.000    2.29e+04    2.49e+04
==============================================================================
Omnibus:                      195.511   Durbin-Watson:                   1.971
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              469.901
Skew:                           1.206   Prob(JB):                    9.17e-103
Kurtosis:                       5.713   Cond. No.                         295.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

 R² = 0.746, todėl apie 75 % medicininių išlaidų (charges) variacijos paaiškinama pasirinktais kintamaisiais.
 Modelis kaip visuma yra statistiškai reikšmingas (F testo p < 0.001).

Age, bmi, children ir smoker_flag yra statistiškai reikšmingi kintamieji. 

Kiekvieni papildomi metai didina išlaidas apie 260 vienetų. 

Didesnis BMI taip pat reikšmingai didina išlaidas. 

Kiekvienas papildomas vaikas padidina išlaidas vidutiniškai apie 541 vienetą.

Smoker_flag turi labai stiprų teigiamą poveikį – rūkantys asmenys vidutiniškai patiria apie 23 900 didesnes išlaidas nei nerūkantys.

Normalumo testai rodo, kad liekanos nėra visiškai normalios, tačiau Durbin–Watson reikšmė (~1.93) rodo, kad autokoreliacijos problemos nėra.

In [5]:
from sklearn.metrics import r2_score as r2
from sklearn.metrics import mean_absolute_error as mae

print(f"Train R2: {r2(y_train, model.predict(X_train))}")
print(f"Train MAE: {mae(y_train, model.predict(X_train))}")
print(f"Validation R2: {r2(y_valid, model.predict(X_valid))}")
print(f"Validation MAE: {mae(y_valid, model.predict(X_valid))}")


Train R2: 0.7460214272397101
Train MAE: 4290.287560582359
Validation R2: 0.7509040796854571
Validation MAE: 4194.961676288338


Modelio rezultatai rodo, kad nėra per didelio persimokymo (overfitting).

Train R² = 0.746 ir Validation R² = 0.751 yra labai panašūs, todėl modelis panašiai paaiškina duomenų variaciją tiek mokymo, tiek validacijos rinkiniuose.

Train MAE ≈ 4290, o Validation MAE ≈ 4195 taip pat yra artimos reikšmės. Tai rodo, kad modelio prognozių klaida išlieka stabili skirtinguose duomenų rinkiniuose.

Kadangi validacijos rezultatai nėra blogesni už mokymo, modelis gerai generalizuojasi ir nėra aiškių overfitting požymių.

### Final Fit (on all training data) & Test Score

In [20]:
model = sm.OLS(y, X).fit()

print(f"Train R2: {r2(y, model.predict(X))}")
print(f"Train MAE: {mae(y, model.predict(X))}")

Train R2: 0.7475937041957614
Train MAE: 4198.787893587895


Train R² = 0.748 rodo, kad modelis paaiškina apie 75 % medicininių išlaidų variacijos mokymo duomenyse.

Train MAE ≈ 4199 reiškia, kad vidutinė absoliuti prognozės klaida yra apie 4200 dolerių- vidutinis skirtumas tarp prognozuotų ir faktinių išlaidų.

Vien tik train rezultatai neleidžia įvertinti modelio generalizacijos, todėl juos reikia lyginti su validation arba test rinkiniu.

In [7]:
print(f"Test R2: {r2(y_test, model.predict(X_test))}")
print(f"Test MAE: {mae(y_test, model.predict(X_test))}")

Test R2: 0.7589480825048381
Test MAE: 4042.1960795494483


Test R² = 0.759 rodo, kad modelis paaiškina apie 76 % medicininių išlaidų variacijos naujuose, nematytuose duomenyse.

Test MAE ≈ 4042 reiškia, kad vidutinė absoliuti prognozės klaida yra apie 4000 piniginių vienetų.

Kadangi test rezultatai yra panašūs arba net šiek tiek geresni nei train ir validation, modelis gerai generalizuojasi ir nėra aiškių overfitting požymių.

### Cross Validation Loop

**5-fold cross-validation**

KFold(n_splits=5, shuffle=True, random_state=2023) padalija duomenis į 5 dalis. Kiekvieną kartą viena dalis naudojama validacijai, o likusios keturios – modelio mokymui. Shuffle=True užtikrina atsitiktinį duomenų maišymą prieš dalinimą.

Ciklo metu:
– Duomenys padalinami į mokymo (train) ir validacijos (val) dalis pagal fold indeksus.
– OLS modelis apmokomas su mokymo duomenimis.
– Apskaičiuojami validacijos R² ir MAE rodikliai.
– Rezultatai išsaugomi sąrašuose.

Pabaigoje išvedamos:
– Visų 5 foldų R² ir MAE reikšmės.
– Vidurkis ir standartinis nuokrypis.

Vidurkis parodo bendrą modelio veikimą, o standartinis nuokrypis parodo, kiek stabilūs yra rezultatai skirtinguose duomenų padalijimuose.

In [8]:
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score as r2


kf = KFold(n_splits=5, shuffle=True, random_state=2023)

# Create a list to store validation scores for each fold
cv_lm_r2s = []
cv_lm_mae = []

# Loop through each fold in X and y
for train_ind, val_ind in kf.split(X, y):
    # Subset data based on CV folds
    X_train, y_train = X.iloc[train_ind], y.iloc[train_ind]
    X_val, y_val = X.iloc[val_ind], y.iloc[val_ind]
    # Fit the Model on fold's training data
    model = sm.OLS(y_train, X_train).fit()
    # Append Validation score to list 
    cv_lm_r2s.append(r2(y_val, model.predict(X_val),))
    cv_lm_mae.append(mae(y_val, model.predict(X_val),))

print("All Validation R2s: ", [round(x, 3) for x in cv_lm_r2s])
print(f"Cross Val R2s: {round(np.mean(cv_lm_r2s), 3)} +- {round(np.std(cv_lm_r2s), 3)}")

print("All Validation MAEs: ", [round(x, 3) for x in cv_lm_mae])
print(f"Cross Val MAEs: {round(np.mean(cv_lm_mae), 3)} +- {round(np.std(cv_lm_mae), 3)}")

All Validation R2s:  [0.736, 0.752, 0.756, 0.77, 0.714]
Cross Val R2s: 0.745 +- 0.019
All Validation MAEs:  [4363.249, 3995.482, 4385.796, 3958.32, 4418.007]
Cross Val MAEs: 4224.171 +- 202.984


5 fold cross-validation  rodo, kad modelio veikimas yra stabilus.

R² reikšmės svyruoja nuo 0.714 iki 0.770, o vidurkis yra 0.745 su nedideliu standartiniu nuokrypiu (± 0.019). Tai reiškia, kad modelis nuosekliai paaiškina apie 75 % išlaidų variacijos skirtinguose duomenų padalijimuose.

MAE reikšmės svyruoja nuo maždaug 3958 iki 4418, o vidurkis yra apie 4224 su ± 203 nuokrypiu. Tai rodo, kad vidutinė prognozės klaida išlieka panaši skirtinguose folduose.

Nedidelis rezultatų svyravimas rodo, kad modelis yra stabilus ir gerai generalizuojasi.

### Final Fit & Score On Test

In [9]:
model = sm.OLS(y, X).fit()

r2(y_test, model.predict(X_test))

0.7589480825048381

Galutinis modelis apmokytas naudojant visus mokymo duomenis (y ir X), o tada įvertintas test rinkinyje.

Test R² = 0.759 rodo, kad modelis paaiškina apie 76 % medicininių išlaidų variacijos naujuose, nematytuose duomenyse.

Kadangi ši reikšmė yra labai panaši į kryžminės validacijos ir ankstesnių padalijimų rezultatus, modelis laikomas stabiliu ir gerai generalizuojančiu.